# Lab: Empirical Analysis of Biological Networks

---

### Objectives

In this lab we will compare **five networks of the same size** side by side:

| Network | Description |
|---|---|
| **Real PPI** | Yeast (*S. cerevisiae*) protein-protein interactions from STRING-db |
| **Degree-preserving random** | Rewired version of the real network (MCMC edge swaps) |
| **Erdős-Rényi (ER)** | Random graph with same $N$ and $\langle k \rangle$ |
| **Barabási-Albert (BA)** | Scale-free graph with same $N$ and $m$ |
| **Hierarchical** | Iteratively constructed modular network |

For each network we will compute and compare:
- Degree distribution $P(k)$
- Clustering coefficient $C(k)$
- Shortest path length distribution
- Centrality measures (degree, betweenness, closeness)

---

## 0. Setup

In [ ]:
import requests
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from collections import defaultdict
import os

# Consistent color scheme for all five networks
COLORS = {
    'Real PPI':        'red',
    'Degree-pres.':    'orange',
    'Erdős-Rényi':     'blue',
    'Barabási-Albert': 'green',
    'Hierarchical':    'purple'
}

---
## 1. Downloading the Real PPI Network from STRING-db

We use the STRING REST API to retrieve high-confidence protein-protein interactions
for *Saccharomyces cerevisiae* (species ID: **4932**).

We will:
1. Download interactions for a seed set of well-known yeast proteins
2. Expand to their interaction partners (1-hop neighborhood)
3. Filter by confidence score ≥ 700 (high confidence)
4. Extract the largest connected component and subsample to ~1000 nodes for speed

In [ ]:
SPECIES    = 4932   # S. cerevisiae
SCORE_THR  = 700    # high confidence (0-1000)
CACHE_FILE = 'string_yeast_ppi.csv'
TARGET_N   = 1000    # subsample size

def get_string_ids(gene_names, species=SPECIES):
    """Map gene names to STRING identifiers."""
    url = 'https://string-db.org/api/json/get_string_ids'
    params = {
        'identifiers': '\r'.join(gene_names),
        'species':     species,
        'limit':       1,
        'echo_query':  1
    }
    r = requests.post(url, data=params)
    r.raise_for_status()
    return [entry['stringId'] for entry in r.json()]


def get_interaction_partners(string_ids, species=SPECIES, score_thr=SCORE_THR):
    """Retrieve all interaction partners for a list of STRING IDs."""
    url = 'https://string-db.org/api/tsv/interaction_partners'
    params = {
        'identifiers':    '%0d'.join(string_ids),
        'species':        species,
        'required_score': score_thr,
        'limit':          0   # no limit on partners
    }
    r = requests.post(url, data=params)
    r.raise_for_status()
    from io import StringIO
    df = pd.read_csv(StringIO(r.text), sep='\t')
    return df[['preferredName_A', 'preferredName_B', 'score']]


# Seed proteins: well-known yeast hub proteins
SEED_GENES = [
    'CDC28', 'CLN1', 'CLN2', 'CLN3', 'SIC1', 'CDC4', 'CDC20',
    'RAD51', 'RAD52', 'RPA1', 'MSH2', 'MLH1',
    'TUB1', 'TUB2', 'TUB4', 'MYO1', 'ACT1',
    'HSP82', 'SSA1', 'SSB1', 'CDC48',
    'RPB1', 'RPB2', 'TBP',  'GCN4', 'RAP1'
]

if os.path.exists(CACHE_FILE):
    print(f'Loading cached network from {CACHE_FILE} ...')
    df_raw = pd.read_csv(CACHE_FILE)
else:
    print('Querying STRING API ...')
    string_ids = get_string_ids(SEED_GENES)
    print(f'  Mapped {len(string_ids)} seed proteins.')
    df_raw = get_interaction_partners(string_ids)
    df_raw.to_csv(CACHE_FILE, index=False)
    print(f'  Downloaded {len(df_raw)} interactions — cached to {CACHE_FILE}.')

print(f'Raw interactions: {len(df_raw)} edges between',
      pd.unique(df_raw[['preferredName_A','preferredName_B']].values.ravel()).shape[0],
      'proteins.')

In [ ]:
# Build full graph, take largest connected component, subsample
G_full = nx.from_pandas_edgelist(
    df_raw, 'preferredName_A', 'preferredName_B', edge_attr='score'
)

# Largest connected component
lcc_nodes = max(nx.connected_components(G_full), key=len)
G_lcc = G_full.subgraph(lcc_nodes).copy()
print(f'Largest connected component: {G_lcc.number_of_nodes()} nodes, '
      f'{G_lcc.number_of_edges()} edges.')

# Subsample: keep high-degree nodes for a denser, more interesting network
if G_lcc.number_of_nodes() > TARGET_N:
    degrees = dict(G_lcc.degree())
    top_nodes = sorted(degrees, key=degrees.get, reverse=True)[:TARGET_N]
    G_real = G_lcc.subgraph(top_nodes).copy()
    # Recheck connectivity after subsampling
    lcc2 = max(nx.connected_components(G_real), key=len)
    G_real = G_real.subgraph(lcc2).copy()
else:
    G_real = G_lcc.copy()

N = G_real.number_of_nodes()
E = G_real.number_of_edges()
avg_k = 2 * E / N
print(f'\nWorking network: N={N} nodes, E={E} edges, <k>={avg_k:.2f}')

---
## 2. Constructing the Four Synthetic Networks

All synthetic networks are matched to the real PPI in size ($N$ nodes)).

### 2a. Degree-preserving randomization (MCMC edge swaps)

In [ ]:
# Start from a copy of the real network and rewire edges
# nx.double_edge_swap performs MCMC rewiring preserving degree sequence
G_degpres = G_real.copy()
nx.double_edge_swap(G_degpres, nswap=E * 10, max_tries=E * 100)

print(f'Degree-preserving: N={G_degpres.number_of_nodes()}, '
      f'E={G_degpres.number_of_edges()}')

# Sanity check: degree sequences should match
deg_real    = sorted([d for _, d in G_real.degree()], reverse=True)
deg_degpres = sorted([d for _, d in G_degpres.degree()], reverse=True)
print(f'Degree sequences match: {deg_real == deg_degpres}')

### 2b. Erdős-Rényi random network

In [ ]:
# Match N and <k>: p = <k> / (N-1)
p_er = avg_k / (N - 1)
# liczba wezłów, prawdopodobieństwo istnienia krawędzi  
G_er = nx.erdos_renyi_graph(N, p_er, seed=1024)


print(f'Erdős-Rényi: N={G_er.number_of_nodes()}, '
      f'E={G_er.number_of_edges()}')

### 2c. Barabási-Albert scale-free network

In [ ]:
# m = number of edges each new node attaches with
# <k> = 2m for large BA networks, so m = round(<k>/2)
m_ba = max(1, round(avg_k / 2))
G_ba = nx.barabasi_albert_graph(N, m_ba, seed=1024)

print(f'Barabási-Albert: N={G_ba.number_of_nodes()}, '
      f'E={G_ba.number_of_edges()}, '
      f'm={m_ba}')

### 2d. Hierarchical network

We implement the Barabási-Ravasz iterative construction:
- Start with a 4-node fully connected core
- At each level: create 3 replicas, connect their peripheral nodes to the central hub
- Repeat until $N \geq$ target size, then subsample 
-  

#### Komentarze
> Ilość wierzchołków będziemy ustalać, będziemy się całość różnic ilością krawędzi ze wzgledu na to w jaki sposób. Uwaga, losowe sieci nie muszą być zupełne(? - istnieją wszystkie połączenia)  wiec wyciągamy największą spójną skłądową i dla niej liczymy wartości 

> w zależności od sieci musimy wybrać inny layout (w zależności od rodzaju grafu) żeby móc zwizualizować odpowiednio graf 

In [ ]:
def build_hierarchical_network(levels=4):
    """
    Barabasi-Ravasz hierarchical network construction.
    Level 0: 4-node clique
    Level k: 3 copies of level k-1, peripheral nodes connected to central hub
    Sizes: 4 -> 16 -> 64 -> 256 -> 1024
    """
    # Level 0: fully connected 4-node core
    G = nx.complete_graph(4)
    # Node 0 is the central hub at level 0
    hub = 0
    peripheral = [1, 2, 3]

    for level in range(1, levels):
        n_existing = G.number_of_nodes()
        # Create 3 copies of the current graph
        new_graphs = []
        offsets = []
        for i in range(3):
            offset = n_existing + i * n_existing
            offsets.append(offset)
            mapping = {old: old + offset for old in G.nodes()}
            H = nx.relabel_nodes(G, mapping)
            new_graphs.append((H, offset))

        # Add all replica edges
        for H, offset in new_graphs:
            G.add_edges_from(H.edges())

        # Connect peripheral nodes of each replica to central hub
        for H, offset in new_graphs:
            replica_peripheral = [p + offset for p in peripheral]
            for p in replica_peripheral:
                G.add_edge(hub, p)

        # Update peripheral set: all non-hub nodes from the replicas
        peripheral = []
        for H, offset in new_graphs:
            peripheral.extend([node for node in H.nodes() if node != hub + offset - offset])
        # peripheral nodes are all nodes except the central hub
        peripheral = [n for n in G.nodes() if n != hub]

    return G


G_hier_full = build_hierarchical_network(levels=6)
print(f'Hierarchical (full): N={G_hier_full.number_of_nodes()}, '
      f'E={G_hier_full.number_of_edges()}')

# Subsample to match N
if G_hier_full.number_of_nodes() > N:
    degs = dict(G_hier_full.degree())
    top  = sorted(degs, key=degs.get, reverse=True)[:N]
    G_hier = G_hier_full.subgraph(top).copy()
else:
    G_hier = G_hier_full.copy()

print(f'Hierarchical (subsampled): N={G_hier.number_of_nodes()}, '
      f'E={G_hier.number_of_edges()}')

---
## 3. Summary of the Five Networks

### Exercise 1: Network summary table

Compute the following properties for all five networks and display as a table:
- `N` = number of nodes
- `E` = number of edges
- `<k>` = average degree
- `<C>` = average clustering coefficient
- `<d>` = average shortest path length

> **Warning:** average shortest path length is only defined for **connected graphs**.
> If a network is not fully connected, `nx.average_shortest_path_length()` will raise a `NetworkXError`.
> Always reduce to the **Largest Connected Component (LCC)** before computing path lengths.
> For large networks, you might need to use the **sampling approach** + `nx.single_source_shortest_path_length` in case it's too slow.


**Answer questions:**
1. Which network has the highest average clustering coefficient $\langle C \rangle$? Does this match your expectation from the lecture?
2. Compare $\langle d \rangle$ across networks. Which shows the small-world property most strongly?
3. ER and BA have similar $\langle k \rangle$ — why do their $\langle C \rangle$ differ?


In [ ]:
G_er.number_of_nodes()

In [ ]:
networks = {
    'Real PPI':        G_real,
    'Degree-pres.':    G_degpres,
    'Erdős-Rényi':     G_er,
    'Barabási-Albert': G_ba,
    'Hierarchical':    G_hier
}

rows = []
for name, G in networks.items():
    # --- YOUR CODE HERE ---
    n = G.number_of_nodes()
    e = G.number_of_edges()
    avg_k = 1/n * (2 * e)  # 1/n E
    avg_C = np.mean(nx.average_clustering(G=G))   # hint: nx.average_clustering
    avg_d = nx.average_shortest_path_length(
            # we choose biggest subset of graph 
            G = G.subgraph(max(nx.connected_components(G=G), key=len))
        )    # hint: use LCC, eventually sampling + average_shortest_path_length
    # ----------------------

    rows.append({'Network': name, 'N': n, 'E': e,
                 '<k>': round(avg_k,2), '<C>': round(avg_C,3),
                 '<d>': round(avg_d,2)})

df_summary = pd.DataFrame(rows).set_index('Network')
print(df_summary.to_string())

## 4. Visualization of the Five Networks

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(22, 5))

for ax, (name, G) in zip(axes, networks.items()):
    #TODO - here change layout
    # pos = nx.spring_layout(G, seed=1024, k=0.5, method='force')           # initial
    pos = nx.kamada_kawai_layout(G)                         # TODO - tutaj trzeba przerobić jak dokładnie metoda działa bo to jest ciekawe
    degrees = dict(G.degree())
    node_size  = [10 + 3 * degrees[n] for n in G.nodes()]
    node_color = [degrees[n] for n in G.nodes()]

    nx.draw_networkx_edges(G, pos, ax=ax, alpha=0.15,
                           edge_color='gray', width=0.5)
    nc = nx.draw_networkx_nodes(G, pos, ax=ax,
                                node_size=node_size,
                                node_color=node_color,
                                cmap=plt.cm.plasma,
                                alpha=0.85)
    ax.set_title(name, fontsize=11, fontweight='bold',
                 color=COLORS[name])
    ax.axis('off')

plt.suptitle('Five networks — node color/size encodes degree',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---
## 5. Degree Distribution $P(k)$

### Exercise 2: Degree distribution $P(k)$

Plot the degree distribution $P(k)$ for all five networks on **log-log** scale. 

*Hint: function `degree` for a graph.*

**Answer questions:**
1. Which networks show a straight line on the log-log plot? What does this indicate?
2. How does the real PPI compare to the degree-preserving randomization? They have the same degree sequence — so why might the distributions look the same here but differ in other properties?

> UWAGA: liniowa zależność w skali podwójnie logarytmicznej!!! 

In [ ]:
# --- YOUR CODE HERE ---

for name, G in networks.items():
    # obtain degrees array
    degrees = np.array([degree for idx, degree in nx.degree(G)])
    # apply log log

    # plot distribution (density because of log log - i assume continuous ~ dense support )
    sns.histplot(degrees, stat='density', log_scale=(True, True))
    plt.title(f"{name} histogram")
    plt.show()

## 6. Centrality Measures

We compute degree, betweenness (Brandes algorithm), and closeness centrality for the five networks.

In [ ]:
print('Computing centrality measures ...')

centralities = {}
for name, G in networks.items():
    print(f'  {name}...')
    centralities[name] = {
        'degree':      nx.degree_centrality(G),
        'betweenness': nx.betweenness_centrality(G, normalized=True),
        'closeness':   nx.closeness_centrality(G)
    }

print('Done.')

### Exercise 3: Top hub proteins in the real PPI

Identify the top 15 proteins ranked by $C_B(v)$. 

Display the results as a table with the following columns:

| Protein | Betweenness | Degree centrality | Degree |
|---------|-------------|-------------------|--------|
| ...     | ...         | ...               | ...    |

**Answer questions:**
1. Do the top betweenness proteins correspond to known essential yeast genes? (Hint: search for them on SGD: https://www.yeastgenome.org)
2. Is there a strong correlation between degree and betweenness in the real PPI? In the ER network?

> between cetrality po tym optimaliozwac znaleźć pozostałe statystyki i potem to podajemy 
>
> przechodzimy dokładnie przez 3 pierwsze 

In [ ]:
# --- YOUR CODE HERE ---
G_real

betweenness = nx.betweenness_centrality(G_real, normalized=True)
degree_cent = nx.degree_centrality(G_real)
degree_raw = dict(G_real.degree())

data = []
for protein in G_real.nodes():
    data.append({
        'Protein': protein,
        'Betweenness': betweenness[protein],
        'Degree centrality': degree_cent[protein],
        'Degree': degree_raw[protein]
    })

df_hubs = pd.DataFrame(data)
df_top15 = df_hubs.sort_values(by='Betweenness', ascending=False).head(15)
df_top15